# Partial Cross-Entropy Loss for Remote Sensing Image Segmentation
### Point-supervised semantic segmentation using sparse annotations

**Problem:** Standard segmentation requires complete pixel-wise masks, which are expensive to annotate. 
This notebook implements a **Partial Cross-Entropy (pCE)** loss that trains on sparse point labels only.

**Formula:**
$$pCE = \frac{\sum (FocalLoss(pred, GT) \times MASK_{labeled})}{\sum MASK_{labeled}}$$

Where `MASK_labeled` is 1 at annotated points, 0 everywhere else.


## 1. Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

import os
import random
from PIL import Image
import requests
import zipfile
import io
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

## 2. Partial Cross-Entropy Loss Implementation

The key idea: only compute loss at annotated point locations, ignore all other pixels.

In [ ]:
class PartialCrossEntropyLoss(nn.Module):
    """
    Partial Cross-Entropy Loss for point-supervised segmentation.
    
    Only computes loss at labeled pixel locations (where mask=1).
    Unlabeled pixels (mask=0) contribute nothing to the gradient.
    
    Formula:
        pCE = sum(CE(pred, GT) * MASK_labeled) / sum(MASK_labeled)
    
    Args:
        ignore_index (int): Label value used to mark unlabeled pixels. Default: 255
        reduction (str): How to aggregate - 'mean' or 'sum'. Default: 'mean'
    """
    
    def __init__(self, ignore_index=255, reduction='mean'):
        super(PartialCrossEntropyLoss, self).__init__()
        self.ignore_index = ignore_index
        self.reduction = reduction
        
    def forward(self, predictions, targets):
        """
        Args:
            predictions: (B, C, H, W) - raw logits from the network
            targets:     (B, H, W)    - labels; ignore_index at unlabeled pixels
        
        Returns:
            Scalar loss computed only over labeled pixels.
        """
        B, C, H, W = predictions.shape
        
        # Build the binary mask: 1 where pixel IS labeled, 0 otherwise
        labeled_mask = (targets != self.ignore_index).float()  # (B, H, W)
        
        # How many labeled pixels do we have?
        num_labeled = labeled_mask.sum()
        
        # Edge case: no labeled pixels at all in this batch
        if num_labeled == 0:
            return predictions.sum() * 0.0
        
        # Replace ignore_index in targets with 0 so cross_entropy doesn't crash.
        # The loss at these positions will be zeroed out by the mask anyway.
        safe_targets = targets.clone()
        safe_targets[targets == self.ignore_index] = 0
        
        # Per-pixel cross-entropy (no reduction yet)
        # predictions: (B, C, H, W) -> cross_entropy returns (B, H, W)
        per_pixel_loss = F.cross_entropy(
            predictions, 
            safe_targets, 
            reduction='none'
        )  # shape: (B, H, W)
        
        # Mask out unlabeled pixels
        masked_loss = per_pixel_loss * labeled_mask  # (B, H, W)
        
        # Average only over labeled pixels
        if self.reduction == 'mean':
            loss = masked_loss.sum() / (num_labeled + 1e-8)
        else:  # 'sum'
            loss = masked_loss.sum()
            
        return loss


class PartialFocalLoss(nn.Module):
    """
    Partial Focal Loss - same masking principle but with focal weighting.
    Focal loss down-weights easy examples, focuses on hard ones.
    
    FL(p_t) = -alpha * (1 - p_t)^gamma * log(p_t)
    
    Args:
        gamma (float): Focusing parameter. 0 = standard CE. Default: 2.0
        alpha (float): Class balancing factor. Default: 0.25
        ignore_index (int): Unlabeled pixel marker. Default: 255
    """
    
    def __init__(self, gamma=2.0, alpha=0.25, ignore_index=255):
        super(PartialFocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.ignore_index = ignore_index
        
    def forward(self, predictions, targets):
        B, C, H, W = predictions.shape
        
        labeled_mask = (targets != self.ignore_index).float()
        num_labeled = labeled_mask.sum()
        
        if num_labeled == 0:
            return predictions.sum() * 0.0
        
        safe_targets = targets.clone()
        safe_targets[targets == self.ignore_index] = 0
        
        # Compute softmax probabilities
        log_probs = F.log_softmax(predictions, dim=1)   # (B, C, H, W)
        probs = torch.exp(log_probs)                     # (B, C, H, W)
        
        # Gather the probability of the correct class at each pixel
        # safe_targets: (B, H, W) -> unsqueeze to (B, 1, H, W) for gather
        p_t = probs.gather(1, safe_targets.unsqueeze(1)).squeeze(1)  # (B, H, W)
        log_p_t = log_probs.gather(1, safe_targets.unsqueeze(1)).squeeze(1)
        
        # Focal weight
        focal_weight = self.alpha * (1 - p_t) ** self.gamma
        
        # Per-pixel focal loss
        per_pixel_focal = -focal_weight * log_p_t  # (B, H, W)
        
        # Apply labeled mask and average
        masked_focal = per_pixel_focal * labeled_mask
        loss = masked_focal.sum() / (num_labeled + 1e-8)
        
        return loss


# Quick sanity check
print("Testing PartialCrossEntropyLoss...")
pce_loss = PartialCrossEntropyLoss(ignore_index=255)

# Dummy data: batch=2, 3 classes, 4x4 spatial
dummy_pred = torch.randn(2, 3, 4, 4)
dummy_target = torch.zeros(2, 4, 4, dtype=torch.long)
dummy_target[0, 0, 0] = 1  # one labeled pixel, class 1
dummy_target[0, 1, 1] = 2  # one labeled pixel, class 2
dummy_target[1, 2, 2] = 0  # one labeled pixel, class 0
# Fill remaining with ignore index
labeled_pos = torch.zeros(2, 4, 4, dtype=torch.bool)
labeled_pos[0, 0, 0] = True
labeled_pos[0, 1, 1] = True  
labeled_pos[1, 2, 2] = True
dummy_target[~labeled_pos] = 255  # mark all others as unlabeled

loss_val = pce_loss(dummy_pred, dummy_target)
print(f"  pCE Loss value: {loss_val.item():.4f}")
print(f"  Labeled pixels: {labeled_pos.sum().item()} / {2*4*4} total")
assert not torch.isnan(loss_val), "Loss is NaN!"
print("  PASSED\n")

print("Testing PartialFocalLoss...")
pfce_loss = PartialFocalLoss(gamma=2.0)
focal_val = pfce_loss(dummy_pred, dummy_target)
print(f"  Focal Loss value: {focal_val.item():.4f}")
assert not torch.isnan(focal_val), "Focal loss is NaN!"
print("  PASSED")

## 3. Simulated Remote Sensing Dataset

We simulate a remote sensing scene with 3 land-cover classes:
- **Class 0:** Background / bare soil (dark)
- **Class 1:** Vegetation / forest (green tones) 
- **Class 2:** Built-up / urban (bright)

Then we randomly sample sparse **point labels** to mimic real annotation conditions.

In [ ]:
def generate_remote_sensing_scene(height=256, width=256, seed=None):
    """
    Generate a synthetic remote sensing image with ground-truth segmentation mask.
    
    Returns:
        image: (3, H, W) float tensor in [0, 1]
        mask:  (H, W) long tensor with class labels {0, 1, 2}
    """
    if seed is not None:
        np.random.seed(seed)
    
    # --- Build ground truth mask using spatial blobs ---
    mask = np.zeros((height, width), dtype=np.int64)
    
    # Class 1: Vegetation patches (elliptical blobs)
    for _ in range(5):
        cx = np.random.randint(30, width - 30)
        cy = np.random.randint(30, height - 30)
        rx = np.random.randint(20, 60)
        ry = np.random.randint(20, 60)
        Y, X = np.ogrid[:height, :width]
        ellipse = ((X - cx) / rx) ** 2 + ((Y - cy) / ry) ** 2 <= 1
        mask[ellipse] = 1
    
    # Class 2: Built-up areas (rectangular blocks)
    for _ in range(4):
        x1 = np.random.randint(0, width - 40)
        y1 = np.random.randint(0, height - 40)
        x2 = x1 + np.random.randint(15, 40)
        y2 = y1 + np.random.randint(15, 40)
        mask[y1:y2, x1:x2] = 2
    
    # --- Build RGB image that roughly corresponds to the mask ---
    image = np.zeros((3, height, width), dtype=np.float32)
    
    noise = np.random.randn(3, height, width) * 0.05
    
    # Background: brownish-gray
    image[0] += 0.45  # R
    image[1] += 0.38  # G
    image[2] += 0.30  # B
    
    # Vegetation: green
    veg = (mask == 1)
    image[0, veg] = 0.15
    image[1, veg] = 0.55
    image[2, veg] = 0.20
    
    # Built-up: bright gray/white
    built = (mask == 2)
    image[0, built] = 0.72
    image[1, built] = 0.70
    image[2, built] = 0.68
    
    image = np.clip(image + noise, 0, 1)
    
    return torch.tensor(image), torch.tensor(mask)


def sample_point_labels(full_mask, num_points_per_class=10, ignore_index=255):
    """
    Randomly sample point annotations from the full mask.
    All un-sampled pixels are set to ignore_index.
    
    Args:
        full_mask: (H, W) long tensor with true class labels
        num_points_per_class: how many point labels to sample per class
        ignore_index: value for unlabeled pixels
    
    Returns:
        point_mask: (H, W) long tensor; labeled pixels keep class, rest = ignore_index
        num_sampled: total number of point annotations used
    """
    H, W = full_mask.shape
    point_mask = torch.full((H, W), ignore_index, dtype=torch.long)
    
    num_classes = full_mask.max().item() + 1
    total_sampled = 0
    
    for cls in range(num_classes):
        # Find all pixels belonging to this class
        coords = (full_mask == cls).nonzero(as_tuple=False)  # (N, 2)
        
        if len(coords) == 0:
            continue
        
        # Sample without replacement (or as many as available)
        n_sample = min(num_points_per_class, len(coords))
        idx = torch.randperm(len(coords))[:n_sample]
        selected = coords[idx]  # (n_sample, 2)
        
        for r, c in selected:
            point_mask[r, c] = cls
            total_sampled += 1
    
    return point_mask, total_sampled


# Visualise what our data looks like
img_example, mask_example = generate_remote_sensing_scene(256, 256, seed=7)
point_mask_example, n_pts = sample_point_labels(mask_example, num_points_per_class=15)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(img_example.permute(1, 2, 0).numpy())
axes[0].set_title('Synthetic Remote Sensing Image', fontsize=11)
axes[0].axis('off')

cmap3 = ListedColormap(['#8B6914', '#228B22', '#C0C0C0'])
im1 = axes[1].imshow(mask_example.numpy(), cmap=cmap3, vmin=0, vmax=2)
axes[1].set_title('Full Ground-Truth Mask', fontsize=11)
axes[1].axis('off')
legend_patches = [
    mpatches.Patch(color='#8B6914', label='Class 0: Background'),
    mpatches.Patch(color='#228B22', label='Class 1: Vegetation'),
    mpatches.Patch(color='#C0C0C0', label='Class 2: Built-up'),
]
axes[1].legend(handles=legend_patches, loc='lower right', fontsize=7)

# Show point labels as colored dots
axes[2].imshow(img_example.permute(1, 2, 0).numpy())
colors_pts = {0: 'brown', 1: 'lime', 2: 'white'}
for cls in range(3):
    pts = (point_mask_example == cls).nonzero(as_tuple=False)
    if len(pts) > 0:
        axes[2].scatter(pts[:, 1].numpy(), pts[:, 0].numpy(), 
                       c=colors_pts[cls], s=20, edgecolors='black', 
                       linewidths=0.5, label=f'Class {cls}', zorder=5)
axes[2].set_title(f'Point Labels Only ({n_pts} points sampled)', fontsize=11)
axes[2].legend(loc='lower right', fontsize=7)
axes[2].axis('off')

plt.tight_layout()
plt.savefig('data_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

H, W = mask_example.shape
print(f"Image size: {H}x{W}")
print(f"Total pixels: {H*W}")
print(f"Point annotations: {n_pts} ({100*n_pts/(H*W):.2f}% of all pixels)")
for cls in range(3):
    cnt = (mask_example == cls).sum().item()
    print(f"  Class {cls}: {cnt} pixels ({100*cnt/(H*W):.1f}%)")

## 4. Dataset and DataLoader

In [ ]:
class RemoteSensingDataset(Dataset):
    """
    Dataset of synthetic remote sensing scenes.
    
    Returns both:
      - point_mask: sparse point annotations (for training with pCE)
      - full_mask:  complete ground-truth (for evaluation only)
    """
    
    def __init__(self, num_samples=200, img_size=128, 
                 points_per_class=10, ignore_index=255, seed=0):
        super().__init__()
        self.num_samples = num_samples
        self.img_size = img_size
        self.points_per_class = points_per_class
        self.ignore_index = ignore_index
        
        print(f"Generating {num_samples} synthetic scenes...")
        self.images = []
        self.full_masks = []
        self.point_masks = []
        
        for i in tqdm(range(num_samples)):
            img, full_mask = generate_remote_sensing_scene(
                img_size, img_size, seed=seed + i
            )
            point_mask, _ = sample_point_labels(
                full_mask, 
                num_points_per_class=points_per_class,
                ignore_index=ignore_index
            )
            self.images.append(img)
            self.full_masks.append(full_mask)
            self.point_masks.append(point_mask)
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        return {
            'image':      self.images[idx],       # (3, H, W) float
            'point_mask': self.point_masks[idx],  # (H, W) long, sparse
            'full_mask':  self.full_masks[idx],   # (H, W) long, dense
        }


# Build train / val splits
IMG_SIZE = 128

train_dataset = RemoteSensingDataset(
    num_samples=160, img_size=IMG_SIZE, 
    points_per_class=10, seed=0
)
val_dataset = RemoteSensingDataset(
    num_samples=40, img_size=IMG_SIZE, 
    points_per_class=10, seed=1000
)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False, num_workers=0)

print(f"\nTrain samples: {len(train_dataset)}")
print(f"Val   samples: {len(val_dataset)}")
print(f"Batch size: 8")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")

# Verify a batch
batch = next(iter(train_loader))
print(f"\nSample batch shapes:")
print(f"  image:      {batch['image'].shape}")
print(f"  point_mask: {batch['point_mask'].shape}")
print(f"  full_mask:  {batch['full_mask'].shape}")

## 5. Segmentation Network — Lightweight U-Net

We use a compact U-Net with encoder-decoder skip connections. 
Simple enough to train fast, yet representative of real segmentation architectures.

In [ ]:
class DoubleConv(nn.Module):
    """Two consecutive 3x3 conv + BN + ReLU blocks."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """
    Lightweight U-Net for semantic segmentation.
    
    Architecture:
        Encoder: 3 → 32 → 64 → 128  (3 downsampling levels)
        Bottleneck: 128 → 256
        Decoder: 256 → 128 → 64 → 32 (with skip connections)
        Head: 32 → num_classes
    """
    
    def __init__(self, in_channels=3, num_classes=3, base_filters=32):
        super(UNet, self).__init__()
        f = base_filters
        
        # ---- Encoder ----
        self.enc1 = DoubleConv(in_channels, f)        # 3  -> 32
        self.enc2 = DoubleConv(f,           f * 2)    # 32 -> 64
        self.enc3 = DoubleConv(f * 2,       f * 4)    # 64 -> 128
        self.pool = nn.MaxPool2d(2, 2)
        
        # ---- Bottleneck ----
        self.bottleneck = DoubleConv(f * 4, f * 8)    # 128 -> 256
        
        # ---- Decoder ----
        self.up3   = nn.ConvTranspose2d(f * 8, f * 4, 2, stride=2)
        self.dec3  = DoubleConv(f * 8, f * 4)         # skip from enc3
        
        self.up2   = nn.ConvTranspose2d(f * 4, f * 2, 2, stride=2)
        self.dec2  = DoubleConv(f * 4, f * 2)         # skip from enc2
        
        self.up1   = nn.ConvTranspose2d(f * 2, f,     2, stride=2)
        self.dec1  = DoubleConv(f * 2, f)             # skip from enc1
        
        # ---- Output head ----
        self.out_conv = nn.Conv2d(f, num_classes, 1)
        
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)           # (B, 32,  H,   W)
        e2 = self.enc2(self.pool(e1))  # (B, 64,  H/2, W/2)
        e3 = self.enc3(self.pool(e2))  # (B, 128, H/4, W/4)
        
        # Bottleneck
        b  = self.bottleneck(self.pool(e3))  # (B, 256, H/8, W/8)
        
        # Decoder with skip connections
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))   # (B, 128, H/4, W/4)
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))  # (B, 64,  H/2, W/2)
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))  # (B, 32,  H,   W)
        
        return self.out_conv(d1)  # (B, num_classes, H, W)


# Verify the model
model_test = UNet(in_channels=3, num_classes=3)
dummy_in = torch.randn(2, 3, 128, 128)
dummy_out = model_test(dummy_in)
print(f"UNet output shape: {dummy_out.shape}  ← should be (2, 3, 128, 128)")

total_params = sum(p.numel() for p in model_test.parameters())
trainable_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f"Total parameters:    {total_params:,}")
print(f"Trainable parameters:{trainable_params:,}")

## 6. Evaluation Metrics

In [ ]:
def compute_metrics(pred_logits, true_masks, num_classes=3):
    """
    Compute pixel accuracy and mean IoU over all classes.
    
    Args:
        pred_logits: (B, C, H, W) raw logits
        true_masks:  (B, H, W) ground-truth labels (no ignore_index — full mask)
    
    Returns:
        pixel_acc (float), mean_iou (float), per_class_iou (list)
    """
    pred_classes = pred_logits.argmax(dim=1)  # (B, H, W)
    
    # Pixel accuracy
    correct = (pred_classes == true_masks).sum().item()
    total   = true_masks.numel()
    pixel_acc = correct / total
    
    # Per-class IoU
    iou_list = []
    for cls in range(num_classes):
        pred_cls = (pred_classes == cls)
        true_cls = (true_masks == cls)
        
        intersection = (pred_cls & true_cls).sum().item()
        union        = (pred_cls | true_cls).sum().item()
        
        if union == 0:
            iou = float('nan')  # class not present in this batch
        else:
            iou = intersection / union
        iou_list.append(iou)
    
    # Mean IoU ignoring NaN classes
    valid_ious = [v for v in iou_list if not np.isnan(v)]
    mean_iou = np.mean(valid_ious) if valid_ious else 0.0
    
    return pixel_acc, mean_iou, iou_list


@torch.no_grad()
def evaluate(model, loader, device, num_classes=3):
    """Run full evaluation on a DataLoader. Returns avg pixel_acc and mean_iou."""
    model.eval()
    all_acc, all_iou = [], []
    
    for batch in loader:
        images    = batch['image'].to(device)
        full_mask = batch['full_mask'].to(device)
        
        logits = model(images)
        acc, miou, _ = compute_metrics(logits, full_mask, num_classes)
        all_acc.append(acc)
        all_iou.append(miou)
    
    return np.mean(all_acc), np.mean(all_iou)


print("Evaluation functions defined. Testing with random predictions...")
rand_logits = torch.randn(4, 3, 128, 128)
rand_mask   = torch.randint(0, 3, (4, 128, 128))
acc, miou, per_cls = compute_metrics(rand_logits, rand_mask)
print(f"  Pixel Acc (random): {acc:.3f}  (expect ~0.33 for 3 classes)")
print(f"  Mean IoU  (random): {miou:.3f}")

## 7. Training Function

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    loss_fn,
    num_epochs=20,
    lr=1e-3,
    device=DEVICE,
    exp_name="experiment"
):
    """
    Full training loop with validation and logging.
    
    Returns:
        history: dict with train_loss, val_acc, val_miou per epoch
    """
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    # Learning rate scheduler: reduce by factor 0.5 if val_miou stagnates
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=4, verbose=False
    )
    
    history = {
        'train_loss': [], 
        'val_acc': [], 
        'val_miou': [],
        'lr': []
    }
    
    best_miou = 0.0
    best_weights = None
    
    print(f"\n{'='*55}")
    print(f"  Training: {exp_name}")
    print(f"  Epochs: {num_epochs}  |  LR: {lr}  |  Device: {device}")
    print(f"{'='*55}")
    print(f"{'Epoch':>6}  {'Train Loss':>11}  {'Val Acc':>8}  {'Val mIoU':>9}  {'LR':>9}")
    print(f"{'-'*55}")
    
    for epoch in range(1, num_epochs + 1):
        # ---- Training phase ----
        model.train()
        epoch_loss = 0.0
        
        for batch in train_loader:
            images      = batch['image'].to(device)
            point_masks = batch['point_mask'].to(device)  # sparse labels
            
            optimizer.zero_grad()
            logits = model(images)                  # (B, C, H, W)
            loss   = loss_fn(logits, point_masks)   # only labeled pixels
            loss.backward()
            
            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            
            optimizer.step()
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(train_loader)
        
        # ---- Validation phase ----
        val_acc, val_miou = evaluate(model, val_loader, device)
        
        # Update LR scheduler
        scheduler.step(val_miou)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Save history
        history['train_loss'].append(avg_loss)
        history['val_acc'].append(val_acc)
        history['val_miou'].append(val_miou)
        history['lr'].append(current_lr)
        
        # Track best model
        if val_miou > best_miou:
            best_miou = val_miou
            import copy
            best_weights = copy.deepcopy(model.state_dict())
        
        # Print every epoch
        marker = " ← best" if val_miou == best_miou else ""
        print(f"{epoch:>6}  {avg_loss:>11.4f}  {val_acc:>8.4f}  {val_miou:>9.4f}  {current_lr:>9.6f}{marker}")
    
    print(f"{'-'*55}")
    print(f"Best Val mIoU: {best_miou:.4f}")
    
    # Restore best weights
    if best_weights is not None:
        model.load_state_dict(best_weights)
    
    return history

print("Training function ready.")

---
# EXPERIMENTS

## Experiment 1: Effect of Number of Point Annotations

**Hypothesis:** More point labels per class → more supervision signal → better segmentation. But beyond a threshold, adding more points gives diminishing returns.

We train the same U-Net with pCE loss under 4 annotation densities: **5, 10, 20, 40 points per class**.

In [ ]:
# ---- Experiment 1 Setup ----
POINTS_LIST = [5, 10, 20, 40]
NUM_EPOCHS  = 25
LR          = 1e-3
NUM_CLASSES = 3

exp1_results = {}  # points -> history

for n_pts in POINTS_LIST:
    set_seed(42)
    
    # Build dataset with this number of points
    train_ds = RemoteSensingDataset(
        num_samples=160, img_size=IMG_SIZE, 
        points_per_class=n_pts, seed=0
    )
    val_ds = RemoteSensingDataset(
        num_samples=40, img_size=IMG_SIZE, 
        points_per_class=n_pts, seed=1000
    )
    tr_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=0)
    vl_loader = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=0)
    
    model = UNet(in_channels=3, num_classes=NUM_CLASSES)
    loss_fn = PartialCrossEntropyLoss(ignore_index=255)
    
    history = train_model(
        model, tr_loader, vl_loader, loss_fn,
        num_epochs=NUM_EPOCHS, lr=LR, device=DEVICE,
        exp_name=f"Exp1: {n_pts} pts/class"
    )
    exp1_results[n_pts] = history

print("\nExperiment 1 complete!")

In [ ]:
# ---- Plot Experiment 1 Results ----
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Experiment 1: Effect of Point Annotation Density', fontsize=13, fontweight='bold')

colors = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']

for i, (n_pts, hist) in enumerate(exp1_results.items()):
    label = f"{n_pts} pts/class"
    epochs = range(1, len(hist['train_loss']) + 1)
    
    axes[0].plot(epochs, hist['train_loss'], color=colors[i], label=label, linewidth=1.8)
    axes[1].plot(epochs, hist['val_acc'],    color=colors[i], label=label, linewidth=1.8)
    axes[2].plot(epochs, hist['val_miou'],   color=colors[i], label=label, linewidth=1.8)

axes[0].set_title('Training Loss (pCE)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_title('Validation Pixel Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].set_title('Validation Mean IoU')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('mIoU')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('exp1_annotation_density.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print("\nExperiment 1 Summary — Best Val mIoU:")
print(f"{'Points/Class':>14}  {'Best mIoU':>10}  {'Best Acc':>9}")
print("-" * 38)
for n_pts, hist in exp1_results.items():
    best_miou = max(hist['val_miou'])
    best_acc  = max(hist['val_acc'])
    print(f"{n_pts:>14}  {best_miou:>10.4f}  {best_acc:>9.4f}")

## Experiment 2: Standard CE Loss vs Partial CE Loss

**Purpose:** Demonstrate why regular cross-entropy fails on sparse point labels and why pCE is needed.

**Hypothesis:** Standard CE treats unlabeled pixels as class 0 (background), causing the model to be biased toward background. pCE correctly ignores unlabeled pixels, leading to significantly better performance.

In [ ]:
class NaiveCELoss(nn.Module):
    """
    Naive baseline: treats ignore_index pixels as class 0.
    This simulates what happens if you apply standard CE without masking.
    """
    def __init__(self, ignore_index=255):
        super().__init__()
        self.ignore_index = ignore_index
        
    def forward(self, predictions, targets):
        # Replace ignore_index with 0 (wrong! treats unlabeled as background)
        bad_targets = targets.clone()
        bad_targets[targets == self.ignore_index] = 0
        return F.cross_entropy(predictions, bad_targets, reduction='mean')


# Fix the dataset to 10 pts/class for this experiment
set_seed(42)
train_ds2 = RemoteSensingDataset(num_samples=160, img_size=IMG_SIZE, points_per_class=10, seed=0)
val_ds2   = RemoteSensingDataset(num_samples=40,  img_size=IMG_SIZE, points_per_class=10, seed=1000)
tr_loader2 = DataLoader(train_ds2, batch_size=8, shuffle=True,  num_workers=0)
vl_loader2 = DataLoader(val_ds2,   batch_size=8, shuffle=False, num_workers=0)

exp2_results = {}

# --- Condition A: Naive CE (incorrect baseline) ---
set_seed(42)
model_naive = UNet(in_channels=3, num_classes=NUM_CLASSES)
hist_naive = train_model(
    model_naive, tr_loader2, vl_loader2,
    NaiveCELoss(ignore_index=255),
    num_epochs=NUM_EPOCHS, lr=LR, device=DEVICE,
    exp_name="Naive CE (treats unlabeled as class 0)"
)
exp2_results['Naive CE'] = hist_naive

# --- Condition B: Partial CE (correct approach) ---
set_seed(42)
model_pce = UNet(in_channels=3, num_classes=NUM_CLASSES)
hist_pce = train_model(
    model_pce, tr_loader2, vl_loader2,
    PartialCrossEntropyLoss(ignore_index=255),
    num_epochs=NUM_EPOCHS, lr=LR, device=DEVICE,
    exp_name="Partial CE (correct masking)"
)
exp2_results['Partial CE'] = hist_pce

# --- Condition C: Partial Focal Loss ---
set_seed(42)
model_focal = UNet(in_channels=3, num_classes=NUM_CLASSES)
hist_focal = train_model(
    model_focal, tr_loader2, vl_loader2,
    PartialFocalLoss(gamma=2.0, alpha=0.25, ignore_index=255),
    num_epochs=NUM_EPOCHS, lr=LR, device=DEVICE,
    exp_name="Partial Focal Loss (gamma=2.0)"
)
exp2_results['Partial Focal'] = hist_focal

print("\nExperiment 2 complete!")

In [ ]:
# ---- Plot Experiment 2 Results ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Experiment 2: Loss Function Comparison', fontsize=13, fontweight='bold')

style_map = {
    'Naive CE':     {'color': '#e74c3c', 'ls': '--',  'lw': 2.0},
    'Partial CE':   {'color': '#2ecc71', 'ls': '-',   'lw': 2.2},
    'Partial Focal':{'color': '#3498db', 'ls': '-.',  'lw': 2.0},
}

for name, hist in exp2_results.items():
    s = style_map[name]
    epochs = range(1, len(hist['val_miou']) + 1)
    axes[0].plot(epochs, hist['val_acc'],  label=name, **s)
    axes[1].plot(epochs, hist['val_miou'], label=name, **s)

axes[0].set_title('Validation Pixel Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_title('Validation Mean IoU')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('mIoU')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('exp2_loss_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nExperiment 2 Summary:")
print(f"{'Method':>16}  {'Best mIoU':>10}  {'Best Acc':>9}")
print("-" * 42)
for name, hist in exp2_results.items():
    print(f"{name:>16}  {max(hist['val_miou']):>10.4f}  {max(hist['val_acc']):>9.4f}")

## 8. Qualitative Results — Prediction Visualisation

In [ ]:
def visualize_predictions(model, dataset, device, num_samples=3, title="Model Predictions"):
    """
    Show image, ground-truth mask, point labels, and model prediction side-by-side.
    """
    model.eval()
    cmap3 = ListedColormap(['#8B6914', '#228B22', '#C0C0C0'])
    class_names = ['Background', 'Vegetation', 'Built-up']
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(14, 3.5 * num_samples))
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.01)
    
    indices = random.sample(range(len(dataset)), num_samples)
    
    for row, idx in enumerate(indices):
        sample = dataset[idx]
        image      = sample['image'].unsqueeze(0).to(device)
        full_mask  = sample['full_mask'].numpy()
        point_mask = sample['point_mask'].numpy()
        
        with torch.no_grad():
            pred = model(image).squeeze(0).cpu()  # (C, H, W)
        pred_cls = pred.argmax(dim=0).numpy()      # (H, W)
        
        img_np = sample['image'].permute(1, 2, 0).numpy()
        
        # --- Col 0: Input Image ---
        axes[row, 0].imshow(img_np)
        axes[row, 0].set_title('Input Image', fontsize=9)
        axes[row, 0].axis('off')
        
        # --- Col 1: Ground Truth ---
        axes[row, 1].imshow(full_mask, cmap=cmap3, vmin=0, vmax=2)
        axes[row, 1].set_title('Ground Truth', fontsize=9)
        axes[row, 1].axis('off')
        
        # --- Col 2: Point Labels ---
        axes[row, 2].imshow(img_np)
        colors_pts = {0: 'brown', 1: 'lime', 2: 'white'}
        for cls in range(3):
            pts = np.argwhere(point_mask == cls)
            if len(pts) > 0:
                axes[row, 2].scatter(pts[:, 1], pts[:, 0], 
                                   c=colors_pts[cls], s=15,
                                   edgecolors='black', linewidths=0.4)
        labeled_count = (point_mask != 255).sum()
        axes[row, 2].set_title(f'Point Labels ({labeled_count}pts)', fontsize=9)
        axes[row, 2].axis('off')
        
        # --- Col 3: Prediction ---
        axes[row, 3].imshow(pred_cls, cmap=cmap3, vmin=0, vmax=2)
        # Compute IoU for this sample
        iou_vals = []
        for cls in range(3):
            inter = np.logical_and(pred_cls == cls, full_mask == cls).sum()
            union = np.logical_or( pred_cls == cls, full_mask == cls).sum()
            if union > 0:
                iou_vals.append(inter / union)
        miou_sample = np.mean(iou_vals)
        axes[row, 3].set_title(f'Prediction (mIoU={miou_sample:.3f})', fontsize=9)
        axes[row, 3].axis('off')
    
    # Legend
    legend_patches = [
        mpatches.Patch(color='#8B6914', label='Background'),
        mpatches.Patch(color='#228B22', label='Vegetation'),
        mpatches.Patch(color='#C0C0C0', label='Built-up'),
    ]
    fig.legend(handles=legend_patches, loc='lower center', 
               ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.02))
    
    plt.tight_layout()
    plt.savefig(f'qualitative_{title.replace(" ","_")}.png', 
                dpi=150, bbox_inches='tight')
    plt.show()


# Visualise best model (Partial CE)
print("Visualizing Partial CE model predictions...")
visualize_predictions(model_pce, val_dataset, DEVICE, 
                      num_samples=3, title="Partial CE Loss Predictions")

## 9. Final Results Summary

In [ ]:
print("="*60)
print("  FINAL RESULTS SUMMARY")
print("="*60)

print("\n--- Experiment 1: Annotation Density (pCE Loss) ---")
print(f"{'Points/Class':>14}  {'Best mIoU':>10}  {'Best Acc':>9}  {'Δ mIoU':>8}")
print("-" * 48)
baseline_miou = None
for n_pts, hist in exp1_results.items():
    bm = max(hist['val_miou'])
    ba = max(hist['val_acc'])
    if baseline_miou is None:
        baseline_miou = bm
        delta = "-"
    else:
        delta = f"+{bm - baseline_miou:.4f}"
    print(f"{n_pts:>14}  {bm:>10.4f}  {ba:>9.4f}  {delta:>8}")

print("\n--- Experiment 2: Loss Function Comparison (10 pts/class) ---")
print(f"{'Method':>16}  {'Best mIoU':>10}  {'Best Acc':>9}")
print("-" * 42)
for name, hist in exp2_results.items():
    print(f"{name:>16}  {max(hist['val_miou']):>10.4f}  {max(hist['val_acc']):>9.4f}")

print("\n--- Key Findings ---")
pce_miou   = max(exp2_results['Partial CE']['val_miou'])
naive_miou = max(exp2_results['Naive CE']['val_miou'])
improvement = (pce_miou - naive_miou) / naive_miou * 100
print(f"  pCE vs Naive CE: +{pce_miou - naive_miou:.4f} mIoU ({improvement:.1f}% relative improvement)")

pts5  = max(exp1_results[5]['val_miou'])
pts40 = max(exp1_results[40]['val_miou'])
print(f"  5 pts → 40 pts per class: +{pts40 - pts5:.4f} mIoU")
print("="*60)